# Dataset Raw de la Bundesliga - EDA

> **Liga:** Bundesliga&nbsp;&nbsp;◦&nbsp;&nbsp;**Temporadas:** 2014-15 → 2023-24&nbsp;&nbsp;◦&nbsp;&nbsp;**Fuente:** football-data.co.uk

## Objetivos
- Comprender la estructura del dataset y la estabilidad del esquema entre temporadas.
- Evaluar la calidad e integridad de los datos (identificador de liga, valores nulos, duplicados y rangos inválidos).
- Identificar posibles riesgos antes de avanzar a la fase de limpieza de datos.

## Estructura del Notebook

| # | Sección | Objetivo |
|---|---------|----------|
| 0 | Entorno y configuración | Librerías y rutas del proyecto |
| 1 | Preparación y organización | Ficheros disponibles y diccionario de temporadas |
| 2 | Consistencia del esquema | Dimensiones, core dataset y tipos de dato |
| 3 | Calidad de datos | Completitud por variable y temporada |
| 4 | Validaciones de integridad | Coherencia, duplicados y valores inválidos |
| 5 | Variables de cuotas | Cobertura y rangos de odds |
| 6 | Conclusiones | Síntesis y consideraciones para limpieza |
| 7 | Exportación | Core dataset validado en Parquet |

---
##

## 0) Entorno y configuración

En esta sección se configuran las dependencias, librerías y parámetros globales necesarios para garantizar la reproducibilidad del análisis.

### 0.1 Imports y configuración global

Carga de librerías estándar, módulos propios y definición de las rutas principales del proyecto.

In [30]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Configuración de rutas del proyecto
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOT = PROJECT_ROOT / "config"
with open(CONFIG_ROOT / "leagues.json") as f:
    ALL_LEAGUES = json.load(f)

# Importación de funciones propias
from src.analysis import check_name_consistency, group_columns

### 0.2 Configuración específica de la liga

Definición de las constantes y rutas concretas para el dataset que se va a analizar.

In [31]:
LEAGUE = "bundesliga" 
FILE_PREFIX = LEAGUE

RAW_DIR = PROJECT_ROOT / "data" / "raw" / LEAGUE
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / LEAGUE
GLOB_PATTERN = f"{FILE_PREFIX}_*_raw.csv"

CORE_DIR = PROCESSED_DIR / "core_validated.parquet"
METADATA_DIR = PROCESSED_DIR / "core_schema.json"

---
##

## 1) Preparación y organización de datos

En esta fase se identifican y organizan los datos por temporada con el fin de establecer una estructura coherente previa al análisis estructural.


### 1.1 Carga de datos y verificación inicial

Se identifican los archivos CSV disponibles y se comprueba la disponibilidad de temporadas suficientes para el análisis.

In [32]:
csv_files = sorted(RAW_DIR.glob(GLOB_PATTERN))

if len(csv_files) < 2:
    raise ValueError(f'Se esperaban al menos 2 ficheros en {RAW_DIR}, encontrados {len(csv_files)}')

print(f"Directorio analizado: {RAW_DIR}")
print(f"Ficheros detectados: {len(csv_files)}")

Directorio analizado: /Users/jorgepais/Desktop/kraken/football-analytics/data/raw/bundesliga
Ficheros detectados: 10


### 1.2 Inspección de ficheros disponibles

Se listan los ficheros detectados para verificar las temporadas disponibles antes de su carga.

In [33]:
names = [p.name for p in csv_files]

HEAD_N = 5
print("Muestra de ficheros:")

if len(names) <= 2 * HEAD_N:
    for n in names:
        print(f"  - {n}")
else:
    for n in names[:HEAD_N]:
        print(f"  - {n}")
    print(f"  ... ({len(names) - 2 * HEAD_N} ficheros omitidos) ...")
    for n in names[-HEAD_N:]:
        print(f"  - {n}")

Muestra de ficheros:
  - bundesliga_2014_15_raw.csv
  - bundesliga_2015_16_raw.csv
  - bundesliga_2016_17_raw.csv
  - bundesliga_2017_18_raw.csv
  - bundesliga_2018_19_raw.csv
  - bundesliga_2019_20_raw.csv
  - bundesliga_2020_21_raw.csv
  - bundesliga_2021_22_raw.csv
  - bundesliga_2022_23_raw.csv
  - bundesliga_2023_24_raw.csv


### 1.3 Construcción del diccionario de temporadas

Se construye un diccionario que asocia cada temporada con su correspondiente DataFrame.

In [34]:
dfs_all = {}

for f in csv_files:
    season = f.stem.replace('_raw', '').replace(f'{FILE_PREFIX}_', '')  
    dfs_all[season] = pd.read_csv(f)

seasons_sorted = sorted(
    dfs_all.keys(),
    key=lambda s: int(s.split('_')[0])
)

n = len(seasons_sorted)
print(f"Temporadas cargadas: {n}  ({seasons_sorted[0]} → {seasons_sorted[-1]})\n")
print("  |  ".join(seasons_sorted))

Temporadas cargadas: 10  (2014_15 → 2023_24)

2014_15  |  2015_16  |  2016_17  |  2017_18  |  2018_19  |  2019_20  |  2020_21  |  2021_22  |  2022_23  |  2023_24


---
##

## 2) Consistencia del esquema de datos

En esta fase se analiza la coherencia en la estructura del dataset entre temporadas, evaluando la estabilidad de sus dimensiones, variables y tipos de dato.

### 2.1 Dimensiones del dataset por temporada

Se comparan filas y columnas de cada temporada para identificar posibles cambios estructurales.

In [35]:
summary = pd.DataFrame(
    [{
        "Temporada": s,
        "Filas": dfs_all[s].shape[0],
        "Columnas": dfs_all[s].shape[1],
    } for s in seasons_sorted]
)

display(summary.style.hide(axis="index"))

rows_unique = summary["Filas"].nunique()
cols_unique = summary["Columnas"].nunique()

if rows_unique == 1:
    print(f"Consistencia en número de filas: todas las temporadas contienen {summary['Filas'].iloc[0]} registros.")
else:
    print("Variación detectada en el número de filas entre temporadas.")

if cols_unique == 1:
    print(f"Consistencia en número de columnas: todas las temporadas contienen {summary['Columnas'].iloc[0]} variables.")
else:
    print("Variación detectada en el número de columnas entre temporadas.")

Temporada,Filas,Columnas
2014_15,306,67
2015_16,306,64
2016_17,306,64
2017_18,306,64
2018_19,306,61
2019_20,306,105
2020_21,306,105
2021_22,306,105
2022_23,306,105
2023_24,306,105


Consistencia en número de filas: todas las temporadas contienen 306 registros.
Variación detectada en el número de columnas entre temporadas.


#### 2.1.1 Diagnóstico si hay inconsistencias en filas

In [36]:
if rows_unique > 1:
    cleaned = []

    for s in seasons_sorted:
        df = dfs_all[s]
        empty_rows = df.isna().all(axis=1).sum()

        if empty_rows:
            df = df.dropna(how="all").copy()

            for col in df.select_dtypes("float64"):
                if df[col].notna().all() and (df[col] % 1 == 0).all():
                    df[col] = df[col].astype("int64")

            dfs_all[s] = df
            cleaned.append((s, empty_rows, len(df)))

    if cleaned:
        print("⚠ Diagnóstico de filas inconsistentes:\n")
        print(f"{'Temporada':<12}  {'Eliminadas':>10}  {'Filas':>6}")
        print(f"{'-'*12}  {'-'*10}  {'-'*6}")
        for s, removed, remaining in cleaned:
            print(f"{s:<12}  {removed:>10}  {remaining:>6}")

    print("\n✓ Todas las temporadas tienen el mismo número de filas.")

else:
    print("✓ Todas las temporadas tienen el mismo número de filas.")

✓ Todas las temporadas tienen el mismo número de filas.


### 2.2 Intersección de columnas comunes

Se obtiene la intersección de columnas comunes a todas las temporadas.

In [37]:
column_sets = [set(dfs_all[s].columns) for s in seasons_sorted]
common_columns = set.intersection(*column_sets)
print(f"Número de columnas comunes a TODAS las temporadas: {len(common_columns)}")

Número de columnas comunes a TODAS las temporadas: 43


### 2.3 Conjunto de variables comunes (core dataset)

Listado de variables comunes a todo el histórico de temporadas.

#### 2.3.1 Resumen global

In [38]:
core_df = pd.DataFrame(sorted(common_columns), columns=["Variable"])
df_class = group_columns(core_df["Variable"])

summary_groups = (
    df_class.groupby("Grupo", as_index=False)
    .agg(**{"Número de variables": ("Variable", "count")})
    .sort_values("Número de variables", ascending=False)
)

display(summary_groups.style.hide(axis="index"))

Grupo,Número de variables
Cuotas de apuestas,21
Estadísticas del partido,12
Resultados y goles,6
Identificación del partido,4


#### 2.3.2 Desagregado por grupos

In [39]:
detail_groups = (
    df_class.sort_values(["Grupo", "Variable"])
    .groupby("Grupo", as_index=False)
    .agg(Variables=("Variable", lambda x: "\n".join(x)))
)

display(
    detail_groups.style
    .set_properties(**{"white-space": "pre-wrap"})
    .hide(axis="index")
)

Grupo,Variables
Cuotas de apuestas,B365A B365D B365H BWA BWD BWH IWA IWD IWH PSA PSCA PSCD PSCH PSD PSH VCA VCD VCH WHA WHD WHH
Estadísticas del partido,AC AF AR AS AST AY HC HF HR HS HST HY
Identificación del partido,AwayTeam Date Div HomeTeam
Resultados y goles,FTAG FTHG FTR HTAG HTHG HTR


### 2.4 Tipos de datos del core dataset

Clasificación de las variables comunes por tipo (numéricas, categóricas y temporales) y validación de sus tipos de dato reales en el dataset.

In [40]:
sample_df = dfs_all[seasons_sorted[0]][sorted(common_columns)]

structure_df = pd.DataFrame({
    "Variable": sample_df.columns,
    "Dtype": sample_df.dtypes.astype(str),
    "Categoría": sample_df.dtypes.apply(
        lambda dt: "Numérica" if pd.api.types.is_numeric_dtype(dt)
        else "Temporal" if pd.api.types.is_datetime64_any_dtype(dt)
        else "Categórica"
    )
})

display(
    structure_df
    .style
    .hide(axis="index")
    .set_table_attributes('style="max-height:400px; overflow-y:auto; display:block;"')
)

print(f"\nPreview de datos — Temporada {seasons_sorted[0]} (5 primeras filas):")
display(sample_df.head(5))

Variable,Dtype,Categoría
AC,int64,Numérica
AF,int64,Numérica
AR,int64,Numérica
AS,int64,Numérica
AST,int64,Numérica
AY,int64,Numérica
AwayTeam,object,Categórica
B365A,float64,Numérica
B365D,float64,Numérica
B365H,float64,Numérica



Preview de datos — Temporada 2014_15 (5 primeras filas):


,AC,AF,AR,AS,AST,AY,AwayTeam,B365A,B365D,B365H,...,PSCD,PSCH,PSD,PSH,VCA,VCD,VCH,WHA,WHD,WHH
0,3,9,0,9,4,2,Wolfsburg,10.0,6.00,1.25,...,6.67,1.29,6.47,1.28,12.00,6.0,1.29,12.0,7.0,1.20
1,4,28,0,11,2,2,Leverkusen,5.0,4.33,1.57,...,4.18,1.75,4.39,1.63,6.00,4.0,1.65,5.0,4.2,1.62
2,5,22,0,13,7,3,Freiburg,3.6,3.40,2.05,...,3.74,2.01,3.63,2.04,4.00,3.5,2.05,3.5,3.3,2.10
3,6,16,0,19,4,2,Hamburg,3.6,3.50,2.00,...,3.62,2.06,3.67,2.04,3.80,3.6,2.05,3.4,3.6,2.05
4,4,15,0,12,5,1,Schalke 04,2.3,3.50,2.90,...,3.60,3.10,3.58,3.15,2.38,3.5,3.20,2.3,3.2,3.20


### 2.5 Cambios en tipos de datos

Detección de columnas cuyo `dtype` (tipo de dato) cambia según la temporada en el core dataset.

In [41]:
drift_candidates = []

for col in common_columns:  
    types_per_season = {s: str(dfs_all[s][col].dtype) for s in seasons_sorted}
    unique_types = set(types_per_season.values())
    
    if len(unique_types) > 1:
        drift_candidates.append({
            "Variable": col,
            **types_per_season
        })

drift_df = pd.DataFrame(drift_candidates)

if len(drift_df) > 0:
    print(f"⚠ Columnas del core con drift en dtype: {len(drift_df)}")
    display(drift_df.set_index("Variable"))
else:
    print("Tipos de datos estables en el core dataset")

Tipos de datos estables en el core dataset


### 2.6 Identificación de columnas numéricas estables

Selección de variables numéricas estables para la posterior validación de valores negativos.

In [42]:
numeric_sets = [
    set(dfs_all[s][sorted(common_columns)].select_dtypes(include="number").columns)
    for s in seasons_sorted
]
core_numeric = sorted(set.intersection(*numeric_sets))
all_numeric_in_core = sorted(set.union(*numeric_sets))
unstable_numeric = sorted(set(all_numeric_in_core) - set(core_numeric))

print(f"Columnas numéricas estables en el core: {len(core_numeric)}")

if len(unstable_numeric) > 0:
    print(f"\n⚠ Columnas del core que son numéricas solo en algunas temporadas: {len(unstable_numeric)}")
    print(f"  {', '.join(unstable_numeric)}")
else:
    print("Todas las columnas numéricas del core son estables")

Columnas numéricas estables en el core: 37
Todas las columnas numéricas del core son estables


---
##

## 3) Calidad de datos: valores nulos

Se analiza la presencia de valores nulos por variable y temporada en el core dataset.

### 3.1 Construcción del resumen de valores nulos

Se consolida la información de valores nulos en una tabla estructurada por temporada y variable.

In [43]:
null_rows = []

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]

    null_counts = df.isna().sum()
    null_pct = (df.isna().mean() * 100).round(2)

    season_nulls = pd.DataFrame({
        "Temporada": s,
        "Casa": df.columns,
        "Nulos": null_counts.values,
        "%": null_pct.values
    })

    null_rows.append(season_nulls)

null_core_df = pd.concat(null_rows, ignore_index=True)

print(f"Nulos detectados en el core dataset: {null_core_df['Nulos'].sum():,}")

Nulos detectados en el core dataset: 534


### 3.2 Columnas con nulos relevantes

Se agregan los resultados para obtener una visión global del nivel de ausencia de datos en cada temporada.

In [44]:
season_summary = (
    null_core_df
    .groupby("Temporada", as_index=False)
    .agg(
        **{
            "% nulos medio": ("%", "mean"),
            "Casas con nulos": ("Nulos", lambda x: (x > 0).sum()),
            "Máx. % nulos": ("%", "max"),
        }
    )
)

season_with_nulls = season_summary.loc[season_summary["Casas con nulos"] > 0]

display(
    season_with_nulls
    .style
    .hide(axis="index")
    .format({
        "% nulos medio": "{:.2f}",
        "Máx. % nulos": "{:.2f}"
    })
)

Temporada,% nulos medio,Casas con nulos,Máx. % nulos
2014_15,0.1,3,1.0
2016_17,0.0,3,0.3
2018_19,0.0,6,0.3
2023_24,3.9,6,53.3


### 3.3 Filtro de nulos relevantes

Se identifican las variables cuyo porcentaje de nulos supera el umbral definido.

In [45]:
THRESHOLD_NULL_PCT = 5.0

null_relevant = (
    null_core_df
    .loc[lambda d: d["%"] > THRESHOLD_NULL_PCT]
    .sort_values(["Temporada", "%"], ascending=[True, False])
    .rename(columns={"Casa": "Casas"})
    .reset_index(drop=True)
)

print(f"Umbral aplicado: > {THRESHOLD_NULL_PCT:.2f}% nulos")
print(f"Filas que superan el umbral: {len(null_relevant)}")

if not null_relevant.empty:
    display(
        null_relevant.style
        .hide(axis="index")
        .format({"%": "{:.2f}"})
    )
else:
    print("Ninguna columna del core supera el umbral")

Umbral aplicado: > 5.0% nulos
Filas que superan el umbral: 3


Temporada,Casas,Nulos,%
2023_24,IWA,163,53.3
2023_24,IWD,163,53.3
2023_24,IWH,163,53.3


### 3.4 Análisis global por variable

Se detectan las variables que presentan mayores niveles de ausencia de datos en el core dataset.

In [46]:
worst_columns = (
    null_core_df
    .groupby("Casa", as_index=False)
    .agg(**{"Máx. % nulos (cualquier temporada)": ("%", "max")})
    .sort_values("Máx. % nulos (cualquier temporada)", ascending=False)
)

worst_columns_filtered = worst_columns.loc[
    worst_columns["Máx. % nulos (cualquier temporada)"] > 0
]

if not worst_columns_filtered.empty:
    display(
        worst_columns_filtered
        .style
        .hide(axis="index")
        .format({"Máx. % nulos (cualquier temporada)": "{:.2f}"})
        .set_table_attributes('style="max-height:300px; overflow-y:auto; display:block;"')
    )
else:
    print("Ninguna casa del core presenta nulos en ninguna temporada")

Casa,Máx. % nulos (cualquier temporada)
IWA,53.3
IWD,53.3
IWH,53.3
BWD,2.9
BWA,2.9
BWH,2.9
PSA,0.3
PSCA,0.3
PSCD,0.3
WHH,0.3


---
##

## 4) Validaciones de integridad

Se realizan comprobaciones básicas de coherencia lógica y estructural sobre el core dataset.


### 4.1 Validación de identificador de liga (`Div`)

Verificación de que la columna `Div` contiene exclusivamente el código esperado para esta competición.

In [47]:
expected_div = next(code for code, name in ALL_LEAGUES.items() if name == LEAGUE)

actual_divs = pd.concat([dfs_all[s]["Div"] for s in seasons_sorted]).unique()

if len(actual_divs) == 1 and actual_divs[0] == expected_div:
    print(f"Validación de Div: valor constante '{expected_div}' ({LEAGUE})")
else:
    print(f"⚠ Valores inesperados en Div: {actual_divs} (esperado: {expected_div})")

Validación de Div: valor constante 'D1' (bundesliga)


### 4.2 Validación de categorías en resultados

Se comprueba que las variables `FTR` (resultado final) y `HTR` (resultado al descanso) contengan únicamente las categorías esperadas: `H`, `D` y `A`.

In [48]:
expected = {"H", "D", "A"}
issues_found = False

for s in seasons_sorted:
    ftr_vals = set(dfs_all[s]["FTR"].dropna().unique())
    htr_vals = set(dfs_all[s]["HTR"].dropna().unique())
    
    if ftr_vals != expected or htr_vals != expected:
        print(f"⚠ {s}: FTR={ftr_vals}, HTR={htr_vals}")
        issues_found = True

if not issues_found:
    print("Validación FTR/HTR: todas las temporadas correctas")

Validación FTR/HTR: todas las temporadas correctas


### 4.3 Consistencia en nombres de equipos

Se valida la consistencia de los nombres de equipos a lo largo de las temporadas del core dataset.

In [49]:
result = check_name_consistency(dfs_all, seasons_sorted, common_columns)

print(f"Equipos únicos: {result['total_teams']}  |  Colisiones: {len(result['collisions'])}\n")

if len(result['collisions']) > 0:
    print("⚠ Colisiones detectadas:")
    display(result['collisions'][["Team_norm", "n_variants", "Variants"]].rename(columns={
        "Team_norm": "Nombre de equipos normalizado",
        "n_variants": "Número de variantes",
        "Variants": "Variantes detectadas"
    })
    .style.hide(axis="index")
)
else:
    print("Nombres de equipos consistentes entre temporadas")

teams_by_season = {}
for s in seasons_sorted:
    df_core = dfs_all[s][sorted(common_columns)]
    teams_by_season[s] = set(df_core["HomeTeam"].unique()) | set(df_core["AwayTeam"].unique())

all_teams = sorted({
    str(team).strip()
    for teams in teams_by_season.values()
    for team in teams
    if pd.notna(team) and str(team).strip() != ""
})
teams_df = pd.DataFrame({"Equipo": all_teams})
display(teams_df.style.hide(axis="index").set_table_attributes('style="max-height:300px; overflow-y:auto; display:block;"'))

Equipos únicos: 28  |  Colisiones: 0

Nombres de equipos consistentes entre temporadas


Equipo
Augsburg
Bayern Munich
Bielefeld
Bochum
Darmstadt
Dortmund
Ein Frankfurt
FC Koln
Fortuna Dusseldorf
Freiburg


### 4.4 Detección de duplicados

Se detectan posibles partidos duplicados utilizando `Date`, `HomeTeam` y `AwayTeam` como identificador del mismo.

In [50]:
duplicates_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]
    dups = df.duplicated(subset=["Date", "HomeTeam", "AwayTeam"], keep=False)
    
    if dups.sum() > 0:
        print(f"⚠ {s}: {dups.sum()} filas duplicadas detectadas")
        display(df[dups][["Date", "HomeTeam", "AwayTeam", "FTR"]])
        duplicates_found = True

if not duplicates_found:
    print("Validación de duplicados: ninguna temporada afectada")

Validación de duplicados: ninguna temporada afectada


### 4.5 Coherencia resultado vs goles

Validación de consistencia entre `FTR` y marcadores finales (`FTHG`, `FTAG`).

In [51]:
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]

    inconsistent = df[
        ((df["FTR"] == "H") & (df["FTHG"] <= df["FTAG"])) |
        ((df["FTR"] == "A") & (df["FTAG"] <= df["FTHG"])) |
        ((df["FTR"] == "D") & (df["FTHG"] != df["FTAG"]))
    ]

    if len(inconsistent) > 0:
        print(f"⚠ {s}: {len(inconsistent)} inconsistencias FTR vs goles")
        display(inconsistent[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]])
        issues_found = True

if not issues_found:
    print("Validación de consistencia FTR: todas las temporadas correctas")

Validación de consistencia FTR: todas las temporadas correctas


### 4.6 Control de valores negativos

Control de calidad para columnas numéricas del core dataset donde no se esperan valores negativos.

In [52]:
negative_summary = []
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]

    for col in core_numeric:
        neg_mask = df[col] < 0
        neg_count = neg_mask.sum()
        
        negative_summary.append({
            "Season": s,
            "Column": col,
            "Negative_values": neg_count
        })
        
        if neg_count > 0:
            print(f"⚠ {s} - {col}: {neg_count} valores negativos")
            display(df[neg_mask][["Date", "HomeTeam", "AwayTeam", col]])
            issues_found = True

if not issues_found:
    print("Validación de valores negativos: todas las columnas correctas")

negative_df = pd.DataFrame(negative_summary)

Validación de valores negativos: todas las columnas correctas


---
##

## 5) Variables de cuotas

Análisis de columnas asociadas a casas de apuestas para verificar su consistencia, integridad y cobertura en el dataset.

### 5.1 Identificación de columnas de casas de apuestas 

Se identifican las columnas de casas de apuestas presentes en todas las temporadas del dataset.

In [53]:
# Prefijos de casas de apuestas según documentación oficial de football-data
bookmaker_prefixes = [
    "1XB", "B365", "BF", "BFD", "BMGM", "BV", "BS", "BW", 
    "CL", "GB", "IW", "LB", "PS", "SO", "SB", "SJ", 
    "SY", "VC", "WH"
]

odds_columns = [col for col in common_columns 
                if any(col.startswith(p) for p in bookmaker_prefixes)]

bookmakers = {}
for col in odds_columns:
    prefix = next(p for p in bookmaker_prefixes if col.startswith(p))
    bookmakers.setdefault(prefix, []).append(col)

print(f"Casas de apuestas en el core: {len(bookmakers)}  |  Columnas de cuotas: {len(odds_columns)}\n")
for book, cols in sorted(bookmakers.items()):
    print(f"  {book:4s} → {', '.join(sorted(cols))}")

Casas de apuestas en el core: 6  |  Columnas de cuotas: 21

  B365 → B365A, B365D, B365H
  BW   → BWA, BWD, BWH
  IW   → IWA, IWD, IWH
  PS   → PSA, PSCA, PSCD, PSCH, PSD, PSH
  VC   → VCA, VCD, VCH
  WH   → WHA, WHD, WHH


### 5.2 Consistencia de mercados por casa de apuestas

Verificación de que cada casa de apuestas tiene las tres columnas esperadas (`H`/`D`/`A`).

In [54]:
issues_found = False
extra_variants = []

for book in bookmakers:
    cols = set(bookmakers[book])
    
    basic = {f"{book}H", f"{book}D", f"{book}A"}
    if not basic.issubset(cols):
        print(f"⚠ {book}: mercado básico H/D/A incompleto")
        issues_found = True
    elif len(cols) > 3:
        extra_variants.append(f"{book} ({', '.join(sorted(cols - basic))})")

if not issues_found:
    print("Todas las casas tienen mercados básicos completos (H/D/A)")
    if extra_variants:
        print(f"\nVariantes adicionales detectadas:")
        for variant in extra_variants:
            print(f"  • {variant}")

Todas las casas tienen mercados básicos completos (H/D/A)

Variantes adicionales detectadas:
  • PS (PSCA, PSCD, PSCH)


### 5.3 Detección de odds inválidas

Identificación de valores fuera de rango esperado (< `1.0` o > `100`).

In [55]:
MIN_VALID_ODD = 1.0   # Cuotas < 1.0 son matemáticamente inválidas
MAX_VALID_ODD = 100.0 # Cuotas > 100 son extremadamente raras en ligas principales

issues_found = False

for s in seasons_sorted:
    df = dfs_all[s]
    for col in odds_columns:
        invalid = ((df[col] < MIN_VALID_ODD) | (df[col] > MAX_VALID_ODD)).sum()
        if invalid > 0:
            print(f"⚠ {s} - {col}: {invalid} cuotas fuera de rango [{MIN_VALID_ODD}, {MAX_VALID_ODD}]")
            issues_found = True

if not issues_found:
    print(f"Todas las cuotas están en el rango válido [{MIN_VALID_ODD}, {MAX_VALID_ODD}]")

Todas las cuotas están en el rango válido [1.0, 100.0]


### 5.4 Cobertura de odds por partido

Detección de partidos sin ninguna cuota disponible en el dataset.

In [56]:
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]
    rows_missing_all_odds = df[odds_columns].isnull().all(axis=1).sum()
    
    if rows_missing_all_odds > 0:
        print(f"⚠ {s}: {rows_missing_all_odds} partido(s) sin cuotas")
        
        missing_odds_mask = df[odds_columns].isnull().all(axis=1)
        display(df[missing_odds_mask][["Date", "HomeTeam", "AwayTeam"]])
        
        issues_found = True

if not issues_found:
    print("Todos los partidos tienen al menos una cuota disponible")

Todos los partidos tienen al menos una cuota disponible


---
##

## 6) Conclusiones del análisis exploratorio

El dataset estudiado comprende **10 temporadas** (2014/15–2023/24) con un total de **3.060 partidos** de la Bundesliga. 

Se observó **consistencia total en el número de filas** entre temporadas (306 partidos cada una). Sin embargo, existe **variabilidad en el número de columnas**, principalmente debido a cambios en la cobertura de variables (especialmente cuotas y campos adicionales). La estructura evoluciona desde **61–67 columnas** (2014/15–2018/19) hasta **105 columnas** (2019/20–2023/24).  

A partir de esta variabilidad se definió un **core dataset de 43 variables comunes**, presente de forma consistente en todas las temporadas, que constituye la base estable para el análisis longitudinal.

El análisis de tipos de datos mostró **estabilidad completa en el core dataset**: las **37 variables numéricas** mantienen tipos consistentes en todas las temporadas, sin detección de drift.


### 6.1 Estructura del core dataset

El core dataset se distribuye en:

| Grupo | Variables |
|-------|-----------|
| Identificación (4) | `Date`, `Div`, `HomeTeam`, `AwayTeam` |
| Resultados (6) | `FTHG`, `FTAG`, `FTR`, `HTHG`, `HTAG`, `HTR` |
| Estadísticas (12) | `HS`, `AS`, `HST`, `AST`, `HF`, `AF`, `HC`, `AC`, `HY`, `AY`, `HR`, `AR` |
| Cuotas (21) | `B365`, `BW`, `IW`, `PS`, `VC`, `WH` (`H`/`D`/`A` + variantes) |


### 6.2 Calidad y completitud

- Se detectaron **534 valores nulos** en el core dataset.
- Los valores nulos se concentran en **variables de cuotas**, especialmente **Interwetten** (`IWA`, `IWD`, `IWH`) con un **53% de nulos en 2023/24**, cuya utilidad real se evaluará en fases posteriores.
- El resto de variables presentan niveles de completitud adecuados.
- La estrategia recomendada es **priorizar casas con mayor cobertura** y menor proporción de nulos en fases posteriores.

### 6.3 Consideraciones para la fase de limpieza

- Conversión de `Date` a formato `datetime`.
- Transformación de `Div`: identificador de liga con valor constante `D1` en todas las temporadas. Se renombrará con un valor más legible en la fase de limpieza.
- Se verificó **consistencia total en nombres de equipos**: **28 equipos únicos** sin colisiones tras normalización.
- Incorporación de validaciones automáticas: categorías válidas (`FTR`/`HTR`), ausencia de duplicados, rangos esperados en cuotas y estadísticas.

En conjunto, el core dataset de **3.060 partidos** y **43 variables** muestra estabilidad estructural, coherencia interna y completitud superior al **97%**. Los únicos ajustes necesarios — nulos concentrados en **Interwetten** y conversión de `Date` — se abordarán en la fase de limpieza.

---
##

## 7) Exportación del core dataset

### 7.1 Montaje del core dataset  

Concatenación de todas las temporadas con las columnas del core dataset.

In [57]:
core_columns = core_df["Variable"].tolist()

for s in seasons_sorted:
    missing = set(core_columns) - set(dfs_all[s].columns)
    if missing:
        raise ValueError(f"⚠ {s}: columnas faltantes en core: {missing}")

df_core_all = pd.concat(
    [dfs_all[s][core_columns] for s in seasons_sorted],
    ignore_index=True
)

print(f"Core dataset montado:")
print(f"  Filas: {len(df_core_all):,} | Columnas: {len(core_columns)}")

Core dataset montado:
  Filas: 3,060 | Columnas: 43


### 7.2 Exportación a Parquet y metadatos

Guardado del core dataset en formato Parquet con archivo JSON de metadatos del esquema.

In [58]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_core_all.to_parquet(CORE_DIR, index=False)

metadata = {
    "num_columns": len(core_columns),
    "num_rows": len(df_core_all),
    "num_seasons": len(seasons_sorted),
    "seasons": seasons_sorted,
    "columns": core_columns,
    "dtypes": {col: str(df_core_all[col].dtype) for col in core_columns}
}

with open(METADATA_DIR, 'w') as f:
    json.dump(metadata, f, indent=2)
    
core_rel = CORE_DIR.relative_to(PROJECT_ROOT)
metadata_rel = METADATA_DIR.relative_to(PROJECT_ROOT)

print(f"Archivos guardados:")
print(f"  · Dataset -> {core_rel}")
print(f"  · Metadatos -> {metadata_rel}")

Archivos guardados:
  · Dataset -> data/processed/bundesliga/core_validated.parquet
  · Metadatos -> data/processed/bundesliga/core_schema.json


---
## 